# Mini Trabalho de DAA - Análise de Redes Ficcionais
### Ano Letivo 2024/25

**Trabalho realizado por:**
* Daniel de Matos Masqueiro, nº 129853
* Dinis Soares Sousa, nº 129820

In [2]:
import random as rnd
import csv
from collections import deque

class Vertex:
    def __init__(self, vertex_id):
        self._vertex_id = vertex_id
    def __hash__(self):
        return hash(self._vertex_id)
    def __str__(self):
        return 'v{0}'.format(self._vertex_id)
    def __eq__(self, vertex):
        if not isinstance(vertex, Vertex): return False
        return self._vertex_id == vertex._vertex_id
    def vertex_id(self):
        return self._vertex_id

class Edge:
    def __init__(self, vertex_1, vertex_2, weight):
        self._vertex_1 = vertex_1
        self._vertex_2 = vertex_2
        self._weight = weight
    def __hash__(self):
        return hash((self._vertex_1, self._vertex_2))
    def __str__(self):
        return 'e({0},{1})w={2}'.format(self._vertex_1, self._vertex_2, self._weight)
    def endpoints(self):
        return (self._vertex_1, self._vertex_2)
    def cost(self):
        return self._weight
    def opposite(self, vertex):
        return self._vertex_2 if vertex == self._vertex_1 else self._vertex_1

class Graph:
    def __init__(self):
        self._adjancencies = {}
        self._vertices = {}
        self._n = 0
        self._m = 0
    def order(self): return self._n
    def size(self): return self._m
    def has_vertex(self, vertex_id): return vertex_id in self._vertices
    def has_edge(self, u_id, v_id):
        if not self.has_vertex(u_id) or not self.has_vertex(v_id): return False
        return self._vertices[v_id] in self._adjancencies[self._vertices[u_id]]
    def insert_vertex(self, vertex_id):
        if not self.has_vertex(vertex_id):
            vertex = Vertex(vertex_id)
            self._vertices[vertex_id] = vertex
            self._adjancencies[vertex] = {}
            self._n += 1
    def insert_edge(self, u_id, v_id, weight=0):
        if not self.has_vertex(u_id): self.insert_vertex(u_id)
        if not self.has_vertex(v_id): self.insert_vertex(v_id)      
        if not self.has_edge(u_id, v_id): self._m += 1
        vertex_u, vertex_v = self._vertices[u_id], self._vertices[v_id]
        e = Edge(vertex_u, vertex_v, weight)    
        self._adjancencies[vertex_u][vertex_v] = e
        self._adjancencies[vertex_v][vertex_u] = e
    def vertices(self): return self._vertices.values()
    def edges(self):
        seen = set()
        for adj_map in self._adjancencies.values():
            for edge in adj_map.values():
                if edge not in seen:
                    yield edge
                    seen.add(edge)
    def incident_edges(self, vertex_id):
        for edge in self._adjancencies[self._vertices[vertex_id]].values():
            yield edge
    def get_vertex(self, vertex_id): return self._vertices.get(vertex_id)

# 1. API CentralityAnalyzer
## 1.1. Análise de Complexidade

A classe CentralityAnalyzer foi concebida para integrar o cálculo de métricas de centralidade e conectividade, utilizando como base a representação de grafos por dicionários encadeados . A complexidade espacial da aplicação é estimada em $O(n + m)$, uma vez que a estrutura de dados interna armazena cada vértice e cada aresta do grafo de forma eficiente. No que concerne à complexidade temporal, o método degree_centrality opera em $O(n)$, enquanto a closeness_centrality e a betweenness_centrality (através do algoritmo de Brandes) apresentam uma complexidade de $O(n \cdot (n + m))$ por exigirem uma travessia BFS a partir de cada um dos $n$ vértices do grafo. A eigenvector_centrality, implementada via Power Iteration, possui uma complexidade de $O(k \cdot m)$ por iteração, onde $k$ é o número de iterações necessárias para atingir a convergência numérica.

## 1.2. Construtor e Estrutura de Dados

> **Nota para Daniel e Dinis:** Devem inserir aqui a vossa estimativa teórica do espaço de memória total utilizado pela aplicação em termos de $n$ e $m$ (explicando o porquê do $O(n+m)$ com base nos atributos da classe que inicializam no método abaixo).

In [3]:
class CentralityAnalyzer:
    def __init__(self, graph):
        self._graph = graph
        self._n = graph.order()
        self._m = graph.size()
        self._vertex_ids = [v.vertex_id() for v in graph.vertices()]

    def bfs(self, source):
        distances = {source: 0}
        predecessors = {source: []}
        sigma = {source: 1}
        queue = deque([source])
        order = []
        while queue:
            u_id = queue.popleft()
            order.append(u_id)
            for edge in self._graph.incident_edges(u_id):
                v_id = edge.opposite(self._graph.get_vertex(u_id)).vertex_id()
                if v_id not in distances:
                    distances[v_id] = distances[u_id] + 1
                    queue.append(v_id)
                    sigma[v_id] = 0
                    predecessors[v_id] = []
                if distances[v_id] == distances[u_id] + 1:
                    sigma[v_id] += sigma.get(u_id, 0)
                    predecessors[v_id].append(u_id)
        return distances, predecessors, sigma, order

    def num_components(self):
        visited = set()
        count = 0
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            if v_id not in visited:
                count += 1
                dist, _, _, _ = self.bfs(v_id)
                visited.update(dist.keys())
        return count

    def largest_component(self):
        visited = set()
        largest_nodes = []
        for v in self._graph.vertices():
            v_id = v.vertex_id()
            if v_id not in visited:
                dist, _, _, _ = self.bfs(v_id)
                component_nodes = list(dist.keys())
                if len(component_nodes) > len(largest_nodes):
                    largest_nodes = component_nodes
                visited.update(component_nodes)
        sub = Graph()
        for v_id in largest_nodes: sub.insert_vertex(v_id)
        for edge in self._graph.edges():
            u_id, v_id = [v.vertex_id() for v in edge.endpoints()]
            if u_id in largest_nodes and v_id in largest_nodes:
                sub.insert_edge(u_id, v_id, edge.cost())
        return sub

# 2. Análise Estrutural dos Grafos
## 2.1. Travessia BFS

> **Nota para Daniel e Dinis:** Descrevam aqui o funcionamento do algoritmo BFS, a estrutura de dados auxiliar utilizada (Fila/deque) e a justificação passo a passo para a complexidade $O(n+m)$ (que não coube no resumo global de 1.1).

## 2.2. Conectividade

> **Nota para Daniel e Dinis:** Apresentem aqui a justificação da complexidade temporal dos métodos de conectividade `num_components()` e `largest_component()`.

In [4]:
def carregar_grafo(caminho):
    g = Graph()
    with open(caminho, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            g.insert_edge(row['Source'], row['Target'], float(row['weight']))
    return g

# Carregamento dos Datasets
g_book1 = carregar_grafo('got_book1.csv')
g_full = carregar_grafo('got_full.csv')

# Execução da Análise
for nome, g in [("Livro 1", g_book1), ("Saga Completa", g_full)]:
    analyzer = CentralityAnalyzer(g)
    n_comp = analyzer.num_components()
    lc = analyzer.largest_component()
    print(f"--- {nome} ---")
    print(f"Conexo: {'Sim' if n_comp == 1 else 'Não'}")
    print(f"Nº Componentes: {n_comp}")
    print(f"Ordem da Maior Componente: {lc.order()}\n")

--- Livro 1 ---
Conexo: Sim
Nº Componentes: 1
Ordem da Maior Componente: 187

--- Saga Completa ---
Conexo: Sim
Nº Componentes: 1
Ordem da Maior Componente: 796



### Conclusões da Secção 2.2

> **Nota para Daniel e Dinis:** Respondam aqui às seguintes questões do enunciado:
> 1. O grafo é conexo? Quantas componentes conexas existem?
> 2. Qual o tamanho da maior componente? Os vértices fora da maior componente têm alguma interpretação no contexto narrativo?
> 3. (Apresentem também o código/resultado para verificar o caminho entre dois personagens específicos à vossa escolha).
> 4. Comparem a estrutura de conectividade. A rede torna-se mais ou menos conexa ao longo dos livros da saga?